<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: #ffffff; margin: 0 0 8px 0; font-size: 2.5em;'>🦀 Rust vs 🐍 Python · Speed Lab</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Free · Kaggle CPU Edition · No GPU needed</h3>
  <p style='color: #aaa; margin: 0; text-align: center;'>Tokenizers • DataFrames • JSON • AI model files • pip vs uv • Write your own Rust | raced live, every result checked</p>
</div>

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white" /></a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white" /></a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white" /></a>
  <a href="https://github.com/cafermutluozkan/free-ai-notebooks"><img src="https://img.shields.io/badge/GitHub-STAR-181717?style=for-the-badge&logo=github&logoColor=white" /></a>
</p>

---

### 🚀 Why is AI moving to Rust?

You write Python, but many of the AI tools you use every day are **Rust under the hood**: Hugging Face `tokenizers`, OpenAI `tiktoken`, `safetensors` (the format every new model ships in), `polars`, `orjson`, `uv` and even Hugging Face's new `hf_xet` downloader. Python is the steering wheel; Rust is the engine.

Instead of just saying it, this notebook **measures it**. It races each Python tool against its Rust twin on the same data, checks that both give **exactly the same answer**, and draws a 16:9 chart of the result. Then you **write and compile your own Rust code** and race it against Python.

| Race | 🐍 Python side | 🦀 Rust side | What it checks |
|---|---|---|---|
| 🔤 **Tokenizer** | pure-Python GPT-2 BPE | HF `tokenizers` (1 core / all cores), OpenAI `tiktoken` | same tokens, one by one |
| 📊 **DataFrames** | `pandas` | `polars` (eager + lazy) | same top-10 best sellers |
| 🧾 **JSON** | `json` | `orjson` | same data back |
| 📦 **AI model file** | PyTorch pickle `.bin` | `safetensors` | same weights (+ fair full-read numbers) |
| ⚡ **Installing packages** | `pip` | `uv` | same packages installed |
| 🦀 **Your own Rust** | count primes / vector search in Python | your Rust code, compiled with **PyO3** | same answer (+ NumPy as a reference) |

**Sizes:** ⚡ Quick (seconds) · ⚖️ Standard · 🏋️ Heavy. Everything runs on the **CPU**, so it does not use your GPU hours.

### ⚡ Quick Start
1. **Settings → Accelerator → None** (no GPU needed; GPU sessions work too)
2. **Settings → Internet → ON**
3. Run **Steps 1-3** = settings, install (Rust compiler + libraries + test data, ~2-3 min), the benchmark engine
4. Run **Step 4** = open the **public link** (`https://....gradio.live`) and press **▶ Run race**

> 💡 **Tips:** Start with **🏆 Scoreboard → Run all tests** to see everything at once. Run a race twice if a number looks odd (the first run can include warm-up). Numbers depend on the machine: on Kaggle you get 4 CPU cores.

---
## Step 1: Settings

Package versions and data files are pinned (version, revision and SHA256). You normally do not need to change anything.

In [ ]:
# Step 1 - Settings. You normally do not need to change anything here.
# Every package and data file is PINNED (version / revision / SHA256), so every run measures the same thing.
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE:
    WORK_ROOT, RESULTS_DIR = Path("/kaggle/temp/speedlab"), Path("/kaggle/working/speedlab_results")
elif ON_COLAB:
    WORK_ROOT, RESULTS_DIR = Path("/content/speedlab"), Path("/content/speedlab_results")
else:
    WORK_ROOT, RESULTS_DIR = Path.cwd() / "speedlab", Path.cwd() / "speedlab_results"

DEFAULT_SIZE = "Standard"     # "Quick" (~1 min for all tests) | "Standard" | "Heavy" (bigger data, needs ~16 GB RAM)
INSTALL_RUST = True           # installs the Rust compiler (~1 min) for the "🦀 Write Rust" tab
RUST_TOOLCHAIN = "stable"
PYO3_VERSION = "0.29.3"       # PyO3 = the bridge that lets Python call Rust functions

# The Rust-powered libraries we race against plain Python (Kaggle already has pandas, numpy, torch,
# tokenizers and safetensors, so those stay as they are)
PIP_PACKAGES = ["polars==2.0.0", "orjson==3.13.0", "uv==0.13.1", "tiktoken==0.14.0", "regex", "pyarrow",
                "psutil", "matplotlib", "gradio>=6.0,<7"]

# Test data: GPT-2 tokenizer + WikiText-2 (Wikipedia articles). (url, size in bytes, SHA256)
HF = "https://huggingface.co"
GPT2_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
WIKITEXT_REVISION = "b08601e04326c79dfdd32d625aee71d232d685c3"
DATA_FILES = {
    "vocab.json": (f"{HF}/openai-community/gpt2/resolve/{GPT2_REVISION}/vocab.json", 1042301,
                   "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"),
    "merges.txt": (f"{HF}/openai-community/gpt2/resolve/{GPT2_REVISION}/merges.txt", 456318,
                   "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"),
    "tokenizer.json": (f"{HF}/openai-community/gpt2/resolve/{GPT2_REVISION}/tokenizer.json", 1355256,
                       "8414cab924d8b9b33013f0d221c5862f365ee9be39c5c2bfae8a5a9e970478a6"),
    "wikitext2_train.parquet": (f"{HF}/datasets/Salesforce/wikitext/resolve/{WIKITEXT_REVISION}/"
                                "wikitext-2-raw-v1/train-00000-of-00001.parquet", 6357543,
                                "e83889baabc497075506f91975be5fac0d45c5290b6b20582c8cd1e853d0c9f7"),
}

DATA_DIR = WORK_ROOT / "data"
RUST_DIR = WORK_ROOT / "speedrust"    # the Rust project (Cargo.toml + src/lib.rs)
RUSTMOD_DIR = WORK_ROOT / "rustmod"   # the compiled module Python imports
ENGINE = Path("speedlab_engine.py").resolve()   # written by Step 3
SHARE = True                          # public gradio.live link
for d in (WORK_ROOT, RESULTS_DIR, DATA_DIR, RUSTMOD_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"Running on {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'this machine'} · work folder {WORK_ROOT} · "
      f"results {RESULTS_DIR}")

---
## Step 2: Install

Installs the Rust-powered libraries (pinned versions), downloads the test data (verified by **size + SHA256**), installs the **Rust compiler** and builds our small Rust module once with **PyO3**. About 2-3 minutes.

In [ ]:
# Step 2 - Install the Rust-powered libraries, the Rust compiler and the test data
import hashlib
import os
import shutil
import subprocess
import time
import urllib.request

START = _lap = time.time()

def lap(what):
    """Prints how long the last part took, so you never have to time the install yourself."""
    global _lap
    print(f"⏱️ {what}: {time.time() - _lap:.0f} s")
    _lap = time.time()

def sh(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True, **kw)

# ---- This machine ----
cores = os.cpu_count()
try:
    ram = int(open("/proc/meminfo").read().split()[1]) / 2**20
except Exception:
    ram = 0
print(f"CPU: {cores} cores · RAM: {ram:.0f} GB · Python {sys.version.split()[0]}")
print("💡 No GPU needed: every test here runs on the CPU (Kaggle: Accelerator → None works too).")

# ---- Python packages (pinned) ----
sh([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *PIP_PACKAGES])
lap("Python packages")

# ---- Test data, verified by size + SHA256 ----
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, size, digest) in DATA_FILES.items():
    path = DATA_DIR / name
    if not (path.exists() and path.stat().st_size == size and sha256(path) == digest):
        print(f"⬇ {name}")
        urllib.request.urlretrieve(url, path)
        if path.stat().st_size != size or sha256(path) != digest:
            path.unlink()
            raise RuntimeError(f"{name}: size/SHA256 mismatch, the download was deleted. Run this cell again.")
    print(f"✅ {name} ({size / 1e6:.1f} MB, SHA256 ok)")

# tiktoken downloads its GPT-2 files on first use: do it now, so the speed test does not include a download
os.environ["TIKTOKEN_CACHE_DIR"] = str(DATA_DIR / "tiktoken")
subprocess.run([sys.executable, "-c", "import tiktoken; tiktoken.get_encoding('gpt2')"], check=False)
lap("Test data")

# ---- Rust compiler + our tiny Rust module ----
CARGO_TOML = f"""[package]
name = "speedrust"
version = "0.1.0"
edition = "2021"

[lib]
name = "speedrust"
crate-type = ["cdylib"]

[dependencies]
pyo3 = "={PYO3_VERSION}"

[profile.release]
opt-level = 3
codegen-units = 1
"""

RUST_SOURCE = """// 🦀 Your Rust code. Change it, then press "Compile & Race".
// Python calls these functions exactly like normal Python functions: speedrust.count_primes(1_000_000)
#[pyo3::pymodule]
mod speedrust {
    use pyo3::prelude::*;

    /// Count the prime numbers below `limit` (the same simple loop as the Python version).
    #[pyfunction]
    fn count_primes(limit: u64) -> u64 {
        let mut count = 0;
        for n in 2..limit {
            let mut is_prime = true;
            let mut d = 2;
            while d * d <= n {
                if n % d == 0 {
                    is_prime = false;
                    break;
                }
                d += 1;
            }
            if is_prime {
                count += 1;
            }
        }
        count
    }

    /// Score every document against the question and return the best `k` (index, score) pairs.
    /// The vectors arrive as raw float32 bytes from NumPy.
    #[pyfunction]
    fn search(query: &[u8], docs: &[u8], dim: usize, k: usize) -> Vec<(usize, f32)> {
        let to_floats = |bytes: &[u8]| -> Vec<f32> {
            bytes.chunks_exact(4).map(|b| f32::from_le_bytes([b[0], b[1], b[2], b[3]])).collect()
        };
        let query = to_floats(query);
        let docs = to_floats(docs);
        let mut scores: Vec<(usize, f32)> = docs
            .chunks_exact(dim)
            .enumerate()
            .map(|(i, row)| (i, row.iter().zip(&query).map(|(a, b)| a * b).sum::<f32>()))
            .collect();
        scores.sort_by(|a, b| b.1.total_cmp(&a.1));
        scores.truncate(k);
        scores
    }
}
"""

CARGO_BIN = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{CARGO_BIN}{os.pathsep}{os.environ['PATH']}"
os.environ.setdefault("CARGO_TERM_COLOR", "never")


def cargo_build(source):
    """Compiles src/lib.rs into a Python module. Yields the compiler output line by line, then True/False."""
    (RUST_DIR / "src").mkdir(parents=True, exist_ok=True)
    (RUST_DIR / "Cargo.toml").write_text(CARGO_TOML)
    (RUST_DIR / "src" / "lib.rs").write_text(source)
    env = dict(os.environ, PYO3_PYTHON=sys.executable, PYO3_BUILD_EXTENSION_MODULE="1")
    p = subprocess.Popen(["cargo", "build", "--release"], cwd=RUST_DIR, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        yield line.rstrip()
    if p.wait() != 0:
        yield False
        return
    release = RUST_DIR / "target" / "release"
    built = next((release / n for n in ("libspeedrust.so", "libspeedrust.dylib", "speedrust.dll")
                  if (release / n).exists()), None)
    if built is None:
        yield "Compiled, but the library file was not found."
        yield False
        return
    shutil.copy2(built, RUSTMOD_DIR / ("speedrust.pyd" if built.suffix == ".dll" else "speedrust.so"))
    yield True


RUST_READY = False
if INSTALL_RUST:
    if not shutil.which("cargo"):
        print("🦀 Installing the Rust compiler (rustup, minimal profile)...")
        sh(["sh", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal "
                        f"--default-toolchain {RUST_TOOLCHAIN}"])
    sh(["rustc", "--version"])
    lap("Rust compiler")
    print("🔨 First build of the Rust module (downloads PyO3 and compiles it once, ~1 min)...")
    for out in cargo_build(RUST_SOURCE):
        if isinstance(out, bool):
            RUST_READY = out
        elif out.strip().startswith(("Compiling", "Finished", "error", "warning")):
            print("  ", out.strip())
    lap("First Rust build")
    print("✅ Rust module ready" if RUST_READY else "⚠ The Rust build failed; the other tests still work.")

check = subprocess.run([sys.executable, "-c",
    "import polars, orjson, tokenizers, safetensors, pandas, numpy, gradio;"
    "print('polars', polars.__version__, '| orjson', orjson.__version__, '| tokenizers', tokenizers.__version__,"
    "'| safetensors', safetensors.__version__, '| pandas', pandas.__version__, '| gradio', gradio.__version__)"],
    capture_output=True, text=True)
print(check.stdout.strip() or check.stderr[-2000:])
print(f"\n✅ Install finished in {(time.time() - START) / 60:.1f} min. Run Step 3 and Step 4.")

---
## Step 3: The benchmark engine

This is the code that runs the races (open source, so you can check that every race is fair). Each race runs in its **own fresh Python process**, so one test never slows down the next. Running this cell only **saves** it as `speedlab_engine.py`.

In [ ]:
%%writefile speedlab_engine.py
# Speed Lab benchmark engine. Every test runs in its own fresh Python process, so one test never slows down the next.
# Usage: python speedlab_engine.py <test> <size> --work <folder>
# It prints one JSON object per line: {"type": "log" | "progress" | "result" | "error", ...}
import argparse, gc, json, os, random, shutil, subprocess, sys, time
from pathlib import Path

SIZES = ("Quick", "Standard", "Heavy")


def emit(kind, **kw):
    print(json.dumps({"type": kind, **kw}), flush=True)


def log(msg):
    emit("log", msg=msg)


def progress(value, msg=""):
    emit("progress", value=round(value, 3), msg=msg)


def timed(fn, *args, **kw):
    t = time.perf_counter()
    out = fn(*args, **kw)
    return out, time.perf_counter() - t


def pick(size, quick, standard, heavy):
    return {"Quick": quick, "Standard": standard, "Heavy": heavy}[size]


# ---------------------------------------------------------------------------
# 1) Tokenizer: pure-Python GPT-2 BPE  vs  Hugging Face tokenizers (Rust)  vs  tiktoken (Rust)
# ---------------------------------------------------------------------------
def bytes_to_unicode():
    # GPT-2 maps every byte to a printable character before BPE (same table as OpenAI's original encoder)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


class PythonBPE:
    """GPT-2 byte-pair encoding written in plain Python (the classic algorithm, with a word cache like the original)."""

    def __init__(self, vocab_path, merges_path):
        import regex
        self.encoder = json.loads(Path(vocab_path).read_text(encoding="utf-8"))
        merges = Path(merges_path).read_text(encoding="utf-8").split("\n")[1:]
        self.ranks = {tuple(m.split()): i for i, m in enumerate(merges) if m.strip()}
        self.byte_encoder = bytes_to_unicode()
        self.cache = {}
        self.pattern = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = list(token)
        while len(word) > 1:
            pairs = [(word[i], word[i + 1]) for i in range(len(word) - 1)]
            best = min(pairs, key=lambda p: self.ranks.get(p, float("inf")))
            if best not in self.ranks:
                break
            merged, i = [], 0
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                    merged.append(word[i] + word[i + 1])
                    i += 2
                else:
                    merged.append(word[i])
                    i += 1
            word = merged
        self.cache[token] = word
        return word

    def encode(self, text):
        ids = []
        for piece in self.pattern.findall(text):
            piece = "".join(self.byte_encoder[b] for b in piece.encode("utf-8"))
            ids.extend(self.encoder[t] for t in self.bpe(piece))
        return ids


def load_lines(data, size):
    import pyarrow.parquet as pq
    lines = [t for t in pq.read_table(data / "wikitext2_train.parquet").column("text").to_pylist() if t.strip()]
    if size == "Quick":
        return lines[: len(lines) // 4]
    if size == "Heavy":
        return lines * 3
    return lines


def test_tokenizer(size, work):
    data = work / "data"
    lines = load_lines(data, size)
    chars = sum(map(len, lines))
    log(f"Text: WikiText-2 (Wikipedia articles), {len(lines):,} lines, {chars / 1e6:.1f} M characters")
    bars, details = [], []

    log("Pure Python BPE: encoding line by line (this is the slow one)...")
    py = PythonBPE(data / "vocab.json", data / "merges.txt")
    t = time.perf_counter()
    py_ids, step = [], max(1, len(lines) // 20)
    for i, line in enumerate(lines):
        py_ids.append(py.encode(line))
        if i % step == 0:
            progress(0.05 + 0.6 * i / len(lines), f"Pure Python: {i:,} / {len(lines):,} lines")
    t_py = time.perf_counter() - t
    tokens = sum(map(len, py_ids))
    bars.append({"label": "Pure Python", "lang": "python", "seconds": t_py})
    log(f"Pure Python: {t_py:.2f} s, {tokens:,} tokens")

    from tokenizers import Tokenizer
    hf = Tokenizer.from_file(str(data / "tokenizer.json"))
    progress(0.7, "Hugging Face tokenizers (Rust), 1 core...")
    hf_ids, t_hf1 = timed(lambda: [hf.encode(line).ids for line in lines])
    bars.append({"label": "HF tokenizers · 1 core", "lang": "rust", "seconds": t_hf1})
    log(f"HF tokenizers, 1 core: {t_hf1:.2f} s")

    progress(0.8, f"Hugging Face tokenizers (Rust), all {os.cpu_count()} cores...")
    batch, t_hfn = timed(getattr(hf, "encode_batch_fast", hf.encode_batch), lines)
    hf_batch_ids = [e.ids for e in batch]
    bars.append({"label": f"HF tokenizers · {os.cpu_count()} cores", "lang": "rust", "seconds": t_hfn})
    log(f"HF tokenizers, all cores: {t_hfn:.2f} s")

    same = [hf_ids == py_ids, hf_batch_ids == py_ids]
    try:
        import tiktoken
        progress(0.9, "tiktoken (OpenAI, Rust), 1 core...")
        enc = tiktoken.get_encoding("gpt2")
        tk_ids, t_tk = timed(lambda: [enc.encode_ordinary(line) for line in lines])
        bars.append({"label": "tiktoken (OpenAI) · 1 core", "lang": "rust", "seconds": t_tk})
        same.append(tk_ids == py_ids)
        log(f"tiktoken, 1 core: {t_tk:.2f} s")
    except Exception as e:
        log(f"tiktoken skipped: {e}")

    details.append(["Text", f"{len(lines):,} lines · {chars / 1e6:.1f} M characters"])
    details.append(["Tokens", f"{tokens:,}"])
    details.append(["Tokens per second (pure Python)", f"{tokens / t_py:,.0f}"])
    details.append(["Tokens per second (fastest Rust)", f"{tokens / min(b['seconds'] for b in bars[1:]):,.0f}"])
    check = (f"✅ Every engine produced the exact same {tokens:,} tokens" if all(same)
             else "⚠️ Token IDs differ between engines (timings are still valid)")
    return {"title": "Tokenizing Wikipedia text (GPT-2 tokenizer)", "bars": bars, "details": details, "check": check,
            "note": "Every LLM tokenizes text before it can read it. Hugging Face `tokenizers` and OpenAI `tiktoken` "
                    "are written in Rust and called from Python."}


# ---------------------------------------------------------------------------
# 2) DataFrames: pandas  vs  polars (Rust)
# ---------------------------------------------------------------------------
def make_sales_csv(path, rows):
    import numpy as np
    import polars as pl
    rng = np.random.default_rng(42)
    countries = ["US", "TR", "DE", "IN", "BR", "JP", "GB", "FR", "LK", "NG", "MX", "KR", "ID", "IT", "ES", "CA",
                 "PK", "EG", "VN", "PH"]
    products = [f"product_{i:03d}" for i in range(200)]
    pl.DataFrame({
        "order_id": np.arange(rows, dtype=np.int64),
        "country": np.array(countries)[rng.integers(0, len(countries), rows)],
        "product": np.array(products)[rng.integers(0, len(products), rows)],
        "price": np.round(rng.uniform(1, 500, rows), 2),
        "qty": rng.integers(1, 10, rows),
        "day": rng.integers(1, 366, rows),
    }).write_csv(path)


def test_dataframe(size, work):
    import pandas as pd
    import polars as pl
    rows = pick(size, 2_000_000, 10_000_000, 30_000_000)
    path = work / "data" / f"sales_{rows}.csv"
    if not path.exists():
        progress(0.05, f"Creating a {rows:,}-row sales CSV (not timed)...")
        make_sales_csv(path, rows)
    log(f"Data: {rows:,} orders, {path.stat().st_size / 1e6:.0f} MB CSV")
    stages = {}

    progress(0.2, "pandas: read CSV...")
    df, stages["pandas read"] = timed(pd.read_csv, path)
    progress(0.4, "pandas: filter + group + sort...")

    def pandas_query(df):
        f = df[df["qty"] > 1]
        f = f.assign(revenue=f["price"] * f["qty"])
        g = f.groupby(["country", "product"], as_index=False).agg(
            revenue=("revenue", "sum"), avg_price=("price", "mean"), orders=("order_id", "count"))
        return g.sort_values("revenue", ascending=False).head(10)

    top_pd, stages["pandas query"] = timed(pandas_query, df)
    del df
    gc.collect()

    progress(0.55, "polars: read CSV...")
    df, stages["polars read"] = timed(pl.read_csv, path)

    def polars_query(frame):
        return (frame.filter(pl.col("qty") > 1)
                .with_columns((pl.col("price") * pl.col("qty")).alias("revenue"))
                .group_by(["country", "product"])
                .agg(pl.col("revenue").sum(), pl.col("price").mean().alias("avg_price"),
                     pl.col("order_id").count().alias("orders"))
                .sort("revenue", descending=True).head(10))

    progress(0.7, "polars: filter + group + sort...")
    top_pl, stages["polars query"] = timed(polars_query, df)
    del df
    gc.collect()

    progress(0.85, "polars lazy: the whole job as one optimized query...")
    top_lazy, t_lazy = timed(lambda: polars_query(pl.scan_csv(path)).collect())

    t_pd = stages["pandas read"] + stages["pandas query"]
    t_pl = stages["polars read"] + stages["polars query"]
    bars = [{"label": "pandas", "lang": "python", "seconds": t_pd},
            {"label": "polars", "lang": "rust", "seconds": t_pl},
            {"label": "polars lazy (one query)", "lang": "rust", "seconds": t_lazy}]
    keys_pd = list(zip(top_pd["country"], top_pd["product"]))
    same = all(keys_pd == list(zip(t["country"].to_list(), t["product"].to_list())) for t in (top_pl, top_lazy))
    rev_ok = all(abs(a - b) < 0.01 * abs(a) + 1 for a, b in zip(top_pd["revenue"], top_pl["revenue"].to_list()))
    details = [[k, f"{v:.2f} s"] for k, v in stages.items()]
    details.append(["Best seller", f"{keys_pd[0][1]} in {keys_pd[0][0]} · ${top_pd['revenue'].iloc[0]:,.0f}"])
    check = ("✅ pandas and polars found the same top-10 best sellers with the same revenue" if same and rev_ok
             else "⚠️ The top-10 lists differ (timings are still valid)")
    return {"title": f"Analyzing {rows / 1e6:g} M sales rows (read CSV + filter + group + sort)", "bars": bars,
            "details": details, "check": check,
            "note": "pandas already runs C/C++ code under the hood, but its group-by mostly uses 1 core. polars is "
                    "written in Rust, uses every core and optimizes the whole query before it runs it."}


# ---------------------------------------------------------------------------
# 3) JSON: Python json  vs  orjson (Rust)
# ---------------------------------------------------------------------------
def test_json(size, work):
    import orjson
    n = pick(size, 100_000, 500_000, 2_000_000)
    progress(0.05, f"Creating {n:,} chat messages (not timed)...")
    rnd = random.Random(7)
    words = "the model writes code fast rust python data train token prompt image video agent free gpu".split()
    records = [{"id": i, "role": rnd.choice(["user", "assistant", "system"]),
                "content": " ".join(rnd.choices(words, k=rnd.randint(8, 40))),
                "tokens": rnd.randint(10, 2000), "score": rnd.random(), "tags": rnd.sample(words, 3),
                "meta": {"model": "demo-8b", "ok": rnd.random() > 0.1}} for i in range(n)]
    stages = {}
    progress(0.3, "Python json: dumps...")
    s, stages["json.dumps"] = timed(json.dumps, records)
    progress(0.45, "Python json: loads...")
    back_py, stages["json.loads"] = timed(json.loads, s)
    progress(0.6, "orjson: dumps...")
    b, stages["orjson.dumps"] = timed(orjson.dumps, records)
    progress(0.75, "orjson: loads...")
    back_rs, stages["orjson.loads"] = timed(orjson.loads, b)
    bars = [{"label": "Python json", "lang": "python", "seconds": stages["json.dumps"] + stages["json.loads"]},
            {"label": "orjson", "lang": "rust", "seconds": stages["orjson.dumps"] + stages["orjson.loads"]}]
    details = [[k, f"{v:.3f} s"] for k, v in stages.items()]
    details.append(["JSON size", f"{len(b) / 1e6:.0f} MB"])
    check = ("✅ Both read back exactly the same data" if back_py == back_rs == records
             else "⚠️ The decoded data differs")
    return {"title": f"Saving + loading {n:,} chat messages as JSON", "bars": bars, "details": details,
            "check": check, "note": "APIs, datasets and agent logs are all JSON. Python's json module is already C "
                                    "code; orjson is Rust."}


# ---------------------------------------------------------------------------
# 4) Model files: pickle (.bin)  vs  safetensors (Rust)
# ---------------------------------------------------------------------------
def best_of(fn, runs=3):
    fn()  # warm-up
    times = []
    for _ in range(runs):
        _, t = timed(fn)
        times.append(t)
        gc.collect()
    return min(times)


def test_model_load(size, work):
    import torch
    from safetensors import safe_open
    from safetensors.torch import load_file, save_file
    gb = pick(size, 0.5, 2, 4)
    count = int(gb * 1024 / 32)  # 4096 x 4096 fp16 = 32 MB per layer
    folder = work / "data" / "model"
    folder.mkdir(parents=True, exist_ok=True)
    progress(0.05, f"Creating a {gb:g} GB model with {count} layers (not timed)...")
    gen = torch.Generator().manual_seed(0)
    state = {f"layers.{i}.weight": torch.randn(4096, 4096, generator=gen).half() for i in range(count)}
    bin_path, st_path = folder / "model.bin", folder / "model.safetensors"
    _, t_save_bin = timed(torch.save, state, bin_path)
    _, t_save_st = timed(save_file, state, str(st_path))
    first = "layers.0.weight"

    progress(0.3, "pickle: torch.load ...")
    t_bin = best_of(lambda: torch.load(bin_path, weights_only=True))
    progress(0.5, "safetensors: load_file ...")
    t_st = best_of(lambda: load_file(str(st_path)))
    progress(0.7, "Opening only the first layer...")
    t_bin_one = best_of(lambda: torch.load(bin_path, weights_only=True)[first])

    def st_one():
        with safe_open(str(st_path), framework="pt") as f:
            return f.get_tensor(first)

    t_st_one = best_of(st_one)

    def touch_all(sd):
        return sum(float(t.float().abs().max()) for t in sd.values())

    progress(0.78, "Load + read every weight once (fair check)...")
    t_bin_all = best_of(lambda: touch_all(torch.load(bin_path, weights_only=True)), runs=1)
    t_st_all = best_of(lambda: touch_all(load_file(str(st_path))), runs=1)
    progress(0.85, "Checking the weights...")
    loaded = load_file(str(st_path))
    same = all(torch.equal(loaded[k], state[k]) for k in list(state)[:4]) and torch.equal(
        torch.load(bin_path, weights_only=True)[first], state[first])
    try:
        t_mmap_one = best_of(lambda: torch.load(bin_path, weights_only=True, mmap=True)[first])
    except Exception:
        t_mmap_one = None
    del state, loaded
    gc.collect()
    shutil.rmtree(folder, ignore_errors=True)

    bars = [{"label": "pickle (.bin)", "lang": "python", "seconds": t_bin_one},
            {"label": "safetensors", "lang": "rust", "seconds": t_st_one}]
    if t_mmap_one is not None:
        bars.append({"label": "pickle + mmap=True (fair check)", "lang": "c", "seconds": t_mmap_one})
    details = [["Model size", f"{gb:g} GB · {count} layers (fp16)"],
               ["Save: pickle / safetensors", f"{t_save_bin:.2f} s / {t_save_st:.2f} s"],
               ["Open the whole model: pickle / safetensors", f"{t_bin:.3f} s / {t_st:.4f} s"],
               ["Open + read every weight once: pickle / safetensors", f"{t_bin_all:.2f} s / {t_st_all:.2f} s"]]
    check = "✅ Both files give back exactly the same weights" if same else "⚠️ The weights differ"
    return {"title": f"Opening a {gb:g} GB AI model and reading its first layer", "bars": bars, "details": details,
            "check": check,
            "note": "safetensors (Hugging Face's Rust file format) maps the file into memory instead of copying it, "
                    "so a layer is read only when it is used. If you read every weight, both take about the same "
                    "time (see the table). The other big win is safety: a pickle file can hide code that runs on "
                    "your computer, a safetensors file cannot."}


# ---------------------------------------------------------------------------
# 5) Installing packages: pip  vs  uv (Rust)
# ---------------------------------------------------------------------------
def test_install(size, work):
    import uv
    pkgs = ["requests", "rich", "pydantic"]
    if size != "Quick":
        pkgs += ["httpx", "fastapi", "jinja2", "click", "typer"]
    if size == "Heavy":
        pkgs += ["scikit-learn", "matplotlib", "sympy", "networkx", "pillow", "tqdm", "pyyaml"]
    root = work / "install_test"
    shutil.rmtree(root, ignore_errors=True)
    log(f"Packages: {' '.join(pkgs)} (+ their dependencies), no cache, into empty folders")

    def run(cmd):
        t = time.perf_counter()
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-1500:])
        return time.perf_counter() - t

    progress(0.1, "pip install ...")
    t_pip = run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--disable-pip-version-check",
                 "--target", str(root / "pip"), *pkgs])
    progress(0.6, "uv pip install ...")
    t_uv = run([uv.find_uv_bin(), "pip", "install", "--quiet", "--no-cache", "--python", sys.executable,
                "--target", str(root / "uv"), *pkgs])

    def dists(p):
        return sorted(d.name.split("-")[0].lower().replace("_", "-") for d in p.glob("*.dist-info"))

    a, b = dists(root / "pip"), dists(root / "uv")
    shutil.rmtree(root, ignore_errors=True)
    bars = [{"label": "pip", "lang": "python", "seconds": t_pip}, {"label": "uv", "lang": "rust", "seconds": t_uv}]
    details = [["Requested", ", ".join(pkgs)], ["Installed (pip / uv)", f"{len(a)} / {len(b)} packages"]]
    check = (f"✅ Both installed the same {len(a)} packages" if a == b
             else f"⚠️ Different package sets: pip {len(a)}, uv {len(b)}")
    return {"title": f"Installing {len(pkgs)} Python packages from scratch", "bars": bars, "details": details,
            "check": check, "note": "uv is a pip replacement written in Rust. Downloads depend on the network, "
                                    "so run it twice if the numbers look odd."}


# ---------------------------------------------------------------------------
# 6) Your own Rust code (the "Write Rust" tab): count primes + vector search
# ---------------------------------------------------------------------------
def count_primes(limit):
    count = 0
    for n in range(2, limit):
        is_prime = True
        d = 2
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 1
        if is_prime:
            count += 1
    return count


def search(query, docs, dim, k):
    scores = []
    for i in range(len(docs) // dim):
        row = docs[i * dim:(i + 1) * dim]
        scores.append((i, sum(a * b for a, b in zip(row, query))))
    scores.sort(key=lambda s: s[1], reverse=True)
    return scores[:k]


def import_rust(work):
    sys.path.insert(0, str(work / "rustmod"))
    try:
        import speedrust
    except ImportError as e:
        raise RuntimeError("Your Rust module is not compiled yet. Open the 🦀 Write Rust tab and press "
                           f"Compile & Race. ({e})")
    return speedrust


def test_rust_primes(size, work):
    rs = import_rust(work)
    if not hasattr(rs, "count_primes"):
        raise RuntimeError("Your Rust code has no `count_primes` function.")
    limit = pick(size, 300_000, 1_000_000, 3_000_000)
    progress(0.1, f"Pure Python: counting primes below {limit:,}...")
    py, t_py = timed(count_primes, limit)
    progress(0.8, "Your Rust code...")
    r, t_rs = timed(rs.count_primes, limit)
    bars = [{"label": "Pure Python", "lang": "python", "seconds": t_py},
            {"label": "Your Rust code", "lang": "rust", "seconds": t_rs}]
    check = f"✅ Both found {py:,} primes" if py == r else f"⚠️ Different answers: Python {py:,}, Rust {r:,}"
    return {"title": f"Counting the prime numbers below {limit:,}", "bars": bars, "check": check,
            "details": [["Primes found", f"{py:,}"], ["Algorithm", "the same simple loop in both languages"]],
            "note": "Same algorithm, line by line. Python reads each line while it runs; Rust is compiled to "
                    "machine code first."}


def test_rust_search(size, work):
    import numpy as np
    from array import array
    rs = import_rust(work)
    if not hasattr(rs, "search"):
        raise RuntimeError("Your Rust code has no `search` function.")
    n, dim, k = pick(size, 20_000, 50_000, 200_000), 384, 5
    progress(0.05, f"Creating {n:,} document embeddings with {dim} numbers each (not timed)...")
    rng = np.random.default_rng(1)
    docs = rng.standard_normal((n, dim), dtype=np.float32)
    docs /= np.linalg.norm(docs, axis=1, keepdims=True)
    target = n // 3
    query = docs[target] + 0.3 * rng.standard_normal(dim, dtype=np.float32) / np.sqrt(dim)
    query = (query / np.linalg.norm(query)).astype(np.float32)
    py_docs, py_query = array("f"), array("f")
    py_docs.frombytes(docs.tobytes())
    py_query.frombytes(query.tobytes())
    doc_bytes, query_bytes = docs.tobytes(), query.tobytes()

    progress(0.15, "Pure Python: scoring every document...")
    py, t_py = timed(search, py_query, py_docs, dim, k)
    progress(0.8, "Your Rust code...")
    r, t_rs = timed(rs.search, query_bytes, doc_bytes, dim, k)
    progress(0.9, "NumPy (C + BLAS), for reference...")

    def numpy_search():
        s = docs @ query
        top = np.argpartition(-s, k)[:k]
        return sorted(((int(i), float(s[i])) for i in top), key=lambda x: -x[1])

    npy, t_np = timed(numpy_search)
    bars = [{"label": "Pure Python", "lang": "python", "seconds": t_py},
            {"label": "Your Rust code", "lang": "rust", "seconds": t_rs},
            {"label": "NumPy (C, reference)", "lang": "c", "seconds": t_np}]
    found = [i for i, _ in py][0] == [i for i, _ in r][0] == npy[0][0] == target
    check = (f"✅ All three found the right document (#{target:,}) first" if found
             else f"⚠️ Top results: Python #{py[0][0]}, Rust #{r[0][0]}, NumPy #{npy[0][0]}")
    return {"title": f"Searching {n:,} documents for the best match (like RAG)", "bars": bars, "check": check,
            "details": [["Documents", f"{n:,} × {dim} numbers ({docs.nbytes / 1e6:.0f} MB)"],
                        ["Top-5 (Rust)", ", ".join(f"#{i} ({s:.3f})" for i, s in r)]],
            "note": "This is what a vector database does in RAG: compare the question with every document. "
                    "NumPy is shown as a reference, because it is also compiled code (C + BLAS)."}


TESTS = {"tokenizer": test_tokenizer, "dataframe": test_dataframe, "json": test_json, "model_load": test_model_load,
         "install": test_install, "rust_primes": test_rust_primes, "rust_search": test_rust_search}

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("test", choices=list(TESTS))
    ap.add_argument("size", choices=SIZES)
    ap.add_argument("--work", required=True)
    a = ap.parse_args()
    try:
        result = TESTS[a.test](a.size, Path(a.work))
        result.update(test=a.test, size=a.size, cores=os.cpu_count())
        emit("result", result=result)
    except Exception as e:
        import traceback
        emit("error", msg=f"{type(e).__name__}: {e}", trace=traceback.format_exc()[-3000:])

---
## Step 4: Launch the Speed Lab

Opens the web page with a **public link**. While this cell runs, the Speed Lab is open; stop it with ■. Every chart is saved to `speedlab_results` (the **Output** panel on the right on Kaggle).

In [ ]:
# Step 4 - Launch the Rust vs Python Speed Lab (web UI with a public link)
import csv
import importlib.metadata as md
import importlib.util
import inspect
import json
import os
import shutil
import subprocess
import time
import zipfile
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.ticker
import matplotlib.pyplot as plt
import gradio as gr

SIZES = ["Quick", "Standard", "Heavy"]
TESTS = {  # label -> (engine test, what it does)
    "🔤 Tokenizer · Python vs HF tokenizers + tiktoken": ("tokenizer",
        "Turns Wikipedia text into GPT-2 tokens, the first thing every LLM does. **Pure-Python BPE** vs "
        "Hugging Face **tokenizers** and OpenAI **tiktoken** (both Rust). Same tokens, checked one by one."),
    "📊 DataFrames · pandas vs polars": ("dataframe",
        "Reads a big sales CSV, filters it, groups it by country + product and sorts it. **pandas** vs "
        "**polars** (Rust). Same top-10 best sellers, checked."),
    "🧾 JSON · json vs orjson": ("json",
        "Saves and loads hundreds of thousands of chat messages as JSON (APIs, datasets, agent logs). "
        "Python's **json** vs **orjson** (Rust)."),
    "📦 AI model files · pickle vs safetensors": ("model_load",
        "Opens a multi-GB model file and reads its first layer. PyTorch **pickle (.bin)** vs Hugging Face "
        "**safetensors** (Rust). The table also shows the fair full-read numbers."),
    "⚡ Installing packages · pip vs uv": ("install",
        "Installs the same Python packages from scratch, no cache. **pip** vs **uv** (Rust)."),
}
RACES = {"🔢 Count primes": "rust_primes", "🔎 Vector search (RAG)": "rust_search"}
SHORT = {"tokenizer": "Tokenizer", "dataframe": "DataFrames", "json": "JSON", "model_load": "Model file (1st layer)",
         "install": "pip vs uv", "rust_primes": "Your Rust: primes", "rust_search": "Your Rust: search"}
ALL_TESTS = [v[0] for v in TESTS.values()] + list(RACES.values())

_spec = importlib.util.spec_from_file_location("speedlab_engine", ENGINE)
engine = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(engine)
PY_CODE = {"rust_primes": inspect.getsource(engine.count_primes), "rust_search": inspect.getsource(engine.search)}

RESULTS_FILE = RESULTS_DIR / "results.json"
LATEST = {}
try:
    for r in json.loads(RESULTS_FILE.read_text()):
        LATEST[r["test"]] = r
except Exception:
    pass
_proc = {"p": None}

# ---------------------------------------------------------------------------
# 1) Running a test (each one in its own fresh Python process)
# ---------------------------------------------------------------------------
def run_engine(test, size):
    cmd = [sys.executable, "-u", str(ENGINE), test, size, "--work", str(WORK_ROOT)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8"))
    _proc["p"] = p
    got_end = False
    try:
        for line in p.stdout:
            try:
                msg = json.loads(line)
            except ValueError:
                msg = {"type": "log", "msg": line.rstrip()}
            got_end = got_end or msg.get("type") in ("result", "error")
            yield msg
        p.wait()
        if not got_end:
            yield {"type": "error", "msg": "⏹ Stopped." if p.returncode and p.returncode < 0
                   else f"The test process ended unexpectedly (exit code {p.returncode}). Is the RAM full? "
                        "Try the Quick size."}
    finally:
        if p.poll() is None:
            p.kill()
        _proc["p"] = None


def stop():
    p = _proc["p"]
    if p is not None and p.poll() is None:
        p.kill()
    return "⏹ Stopped."


def fmt_time(s):
    if s < 0.001:
        return f"{s * 1e6:.0f} µs"
    if s < 1:
        return f"{s * 1000:.0f} ms" if s >= 0.1 else f"{s * 1000:.1f} ms"
    return f"{s:.2f} s" if s < 60 else f"{int(s // 60)} min {s % 60:.0f} s"


def fmt_x(x):
    return f"{x:,.0f}×" if x >= 10 else f"{x:.1f}×"


def summarize(r):
    base = r["bars"][0]
    rust = [b for b in r["bars"] if b["lang"] == "rust"]
    best = min(rust, key=lambda b: b["seconds"])
    r["speedup"] = base["seconds"] / max(best["seconds"], 1e-9)
    r["winner"], r["baseline"] = best["label"], base["label"]
    word = "faster" if r["speedup"] >= 1 else "slower"
    ratio = r["speedup"] if r["speedup"] >= 1 else 1 / r["speedup"]
    r["headline"] = f"{best['label']} is {fmt_x(ratio)} {word} than {base['label']}"
    return r

# ---------------------------------------------------------------------------
# 2) Charts (16:9, ready to drop into a video)
# ---------------------------------------------------------------------------
COLORS = {"python": "#4B8BBE", "rust": "#F74C00", "c": "#8B8FA8"}
BG, FG, SUB = "#0f0c29", "#ffffff", "#c0c0ff"


def _style(fig, ax):
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(colors="#9a9ab0", labelsize=12)
    ax.grid(axis="x", color="#ffffff", alpha=0.08)
    ax.set_axisbelow(True)


def draw_chart(r):
    bars = r["bars"][::-1]
    base = r["bars"][0]["seconds"]
    fig, ax = plt.subplots(figsize=(12, 6.75), dpi=110)
    _style(fig, ax)
    secs = [b["seconds"] for b in bars]
    top = max(secs)
    ax.barh(range(len(bars)), [max(v, top * 0.004) for v in secs], color=[COLORS[b["lang"]] for b in bars],
            height=0.62)  # a tiny minimum width, so a 0.1 ms bar is still visible
    for i, b in enumerate(bars):
        txt = fmt_time(b["seconds"])
        if b is not r["bars"][0]:
            x = base / max(b["seconds"], 1e-9)
            txt += f"   {fmt_x(x)} faster" if x >= 1 else f"   {fmt_x(1 / x)} slower"
        ax.text(b["seconds"] + top * 0.015, i, txt, va="center", color=FG, fontsize=15, fontweight="bold")
    ax.set_yticks(range(len(bars)))
    ax.set_yticklabels([b["label"] for b in bars], color=FG, fontsize=15)
    ax.set_xlim(0, top * 1.5)
    ax.set_xlabel("seconds (shorter bar = faster)", color="#9a9ab0", fontsize=12)
    fig.suptitle(r["headline"], color="#FFD43B" if r["speedup"] >= 1 else FG, fontsize=22, fontweight="bold",
                 x=0.02, ha="left")
    ax.set_title(r["title"], color=SUB, fontsize=14, loc="left", pad=12)
    fig.text(0.99, 0.015, f"TheBuildAI Speed Lab · {r['cores']} CPU cores · {r['size']}", ha="right",
             color="#77779a", fontsize=10)
    used = {b["lang"] for b in bars}
    x = 0.01
    for lang, name in (("python", "Python"), ("rust", "Rust"), ("c", "Reference")):
        if lang in used:
            fig.text(x, 0.015, f"■ {name}", color=COLORS[lang], fontsize=11, fontweight="bold")
            x += 0.04 + 0.0085 * len(name)
    fig.tight_layout(rect=(0, 0.03, 1, 0.94))
    path = RESULTS_DIR / f"{r['test']}_{r['size'].lower()}_{datetime.now():%H%M%S}.png"
    fig.savefig(path, facecolor=BG)
    plt.close(fig)
    return str(path)


def draw_scoreboard():
    rows = [LATEST[t] for t in ALL_TESTS if t in LATEST]
    if not rows:
        return None
    rows = rows[::-1]
    fig, ax = plt.subplots(figsize=(12, 6.75), dpi=110)
    _style(fig, ax)
    x = [max(r["speedup"], 1e-3) for r in rows]
    ax.barh(range(len(rows)), x, color=[COLORS["rust"] if v >= 1 else COLORS["python"] for v in x], height=0.62)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}×"))
    ax.axvline(1, color="#ffffff", alpha=0.35, lw=1)
    for i, (r, v) in enumerate(zip(rows, x)):
        ax.text(v * 1.08, i, f"{fmt_x(v)}  ({r['winner']})", va="center", color=FG, fontsize=13, fontweight="bold")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([SHORT[r["test"]] for r in rows], color=FG, fontsize=15)
    ax.set_xlim(0.5, max(x) * 12)
    ax.set_xlabel("how many times faster than the Python version (log scale)", color="#9a9ab0", fontsize=12)
    fig.suptitle("Rust vs Python: the scoreboard", color="#FFD43B", fontsize=24, fontweight="bold", x=0.02,
                 ha="left")
    ax.set_title(f"Measured live on {rows[0]['cores']} CPU cores, no GPU", color=SUB, fontsize=14, loc="left", pad=12)
    fig.text(0.99, 0.015, "TheBuildAI Speed Lab", ha="right", color="#77779a", fontsize=10)
    fig.tight_layout(rect=(0, 0.03, 1, 0.94))
    path = RESULTS_DIR / "scoreboard.png"
    fig.savefig(path, facecolor=BG)
    plt.close(fig)
    return str(path)


def result_md(r):
    out = [f"### 🦀 {r['headline']}", r["check"], "", "| Engine | Time | vs first |", "|---|---|---|"]
    base = r["bars"][0]["seconds"]
    for b in r["bars"]:
        icon = {"python": "🐍", "rust": "🦀", "c": "⚙️"}[b["lang"]]
        out.append(f"| {icon} {b['label']} | **{fmt_time(b['seconds'])}** | {fmt_x(base / max(b['seconds'], 1e-9))} |")
    if r.get("details"):
        out += ["", "| Detail | Value |", "|---|---|"] + [f"| {k} | {v} |" for k, v in r["details"]]
    out += ["", f"> 💡 {r['note']}"]
    return "\n".join(out)


def save_result(r):
    LATEST[r["test"]] = r
    RESULTS_FILE.write_text(json.dumps(list(LATEST.values()), indent=1))


def run_one(test, size, log_lines):
    """Runs one test; yields ("status", text) while it works, then ("done", result | None, message)."""
    start = time.time()
    for msg in run_engine(test, size):
        kind = msg["type"]
        if kind == "log":
            log_lines.append(msg["msg"])
            yield "status", f"⏳ {time.time() - start:.0f} s · {msg['msg']}"
        elif kind == "progress":
            yield "status", f"⏳ {time.time() - start:.0f} s · {msg['value'] * 100:.0f}% · {msg['msg']}"
        elif kind == "error":
            log_lines.append(msg["msg"] + ("\n" + msg["trace"] if msg.get("trace") else ""))
            yield "done", None, msg["msg"]
            return
        elif kind == "result":
            r = summarize(msg["result"])
            r["chart"] = draw_chart(r)
            r["when"] = datetime.now().isoformat(timespec="seconds")
            save_result(r)
            yield "done", r, f"✅ Done in {time.time() - start:.0f} s"
            return

# ---------------------------------------------------------------------------
# 3) Tab handlers
# ---------------------------------------------------------------------------
def arena(label, size):
    test = TESTS[label][0]
    log = []
    yield None, "", "⏳ Starting a fresh Python process...", ""
    for out in run_one(test, size, log):
        if out[0] == "status":
            yield gr.update(), gr.update(), out[1], "\n".join(log[-200:])
        else:
            r, msg = out[1], out[2]
            yield (r["chart"] if r else None), (result_md(r) if r else f"### ❌ {msg}"), msg, "\n".join(log[-200:])


def rust_toolchain_ok():
    return shutil.which("cargo") is not None


def compile_and_race(code, race, size):
    test = RACES[race]
    log = []
    if not rust_toolchain_ok():
        yield None, "### ❌ The Rust compiler is not installed. Set `INSTALL_RUST = True` and run Steps 1-2.", \
            "No Rust compiler", ""
        return
    yield None, "", "🔨 Compiling your Rust code...", ""
    start, ok = time.time(), False
    for out in cargo_build(code):
        if isinstance(out, bool):
            ok = out
        else:
            log.append(out)
            yield gr.update(), gr.update(), f"🔨 {time.time() - start:.0f} s · {out.strip()[:120]}", "\n".join(log)
    if not ok:
        yield None, ("### ❌ The Rust compiler found a problem\nRead the compiler output below: Rust tells you the "
                     "exact line, what is wrong and often how to fix it. Fix it, or press **↺ Reset Rust code**."), \
            "❌ Compile error", "\n".join(log)
        return
    log.append(f"✅ Compiled in {time.time() - start:.1f} s\n")
    for out in run_one(test, size, log):
        if out[0] == "status":
            yield gr.update(), gr.update(), out[1], "\n".join(log)
        else:
            r, msg = out[1], out[2]
            yield (r["chart"] if r else None), (result_md(r) if r else f"### ❌ {msg}"), msg, "\n".join(log)


def scoreboard_table():
    rows = [LATEST[t] for t in ALL_TESTS if t in LATEST]
    if not rows:
        return "No results yet. Press **▶ Run all tests**, or run tests one by one in the other tabs."
    out = ["| Test | Python side | Rust side | Speed-up | Check |", "|---|---|---|---|---|"]
    for r in rows:
        out.append(f"| {SHORT[r['test']]} | {r['baseline']} · {fmt_time(r['bars'][0]['seconds'])} | {r['winner']} · "
                   f"{fmt_time(min(b['seconds'] for b in r['bars'] if b['label'] == r['winner']))} | "
                   f"**{fmt_x(r['speedup'])}** | {r['check'][:1]} |")
    return "\n".join(out)


def gallery():
    return [LATEST[t]["chart"] for t in ALL_TESTS if t in LATEST and Path(LATEST[t]["chart"]).exists()]


def run_all(size):
    log = []
    tests = ALL_TESTS[:]
    if not rust_toolchain_ok():
        tests = [t for t in tests if t not in RACES.values()]
    elif not any(RUSTMOD_DIR.glob("speedrust.*")):
        for out in cargo_build(RUST_SOURCE):
            if not isinstance(out, bool):
                log.append(out)
    total = time.time()
    for i, test in enumerate(tests, 1):
        for out in run_one(test, size, log):
            if out[0] == "status":
                yield gr.update(), gr.update(), gr.update(), f"[{i}/{len(tests)}] {SHORT[test]} · {out[1]}"
            else:
                if out[1] is None:
                    log.append(f"{SHORT[test]}: {out[2]}")
                    if "Stopped" in out[2]:
                        yield draw_scoreboard(), scoreboard_table(), gallery(), "⏹ Stopped."
                        return
                yield draw_scoreboard(), scoreboard_table(), gallery(), f"[{i}/{len(tests)}] {SHORT[test]} · {out[2]}"
    yield draw_scoreboard(), scoreboard_table(), gallery(), f"🏁 All tests finished in {time.time() - total:.0f} s"


def zip_results():
    files = [p for p in RESULTS_DIR.iterdir() if p.suffix in (".png", ".json")]
    if not files:
        return None
    with open(RESULTS_DIR / "results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["test", "size", "engine", "seconds", "speedup_vs_first", "cores", "when"])
        for r in LATEST.values():
            for b in r["bars"]:
                w.writerow([r["test"], r["size"], b["label"], f"{b['seconds']:.6f}",
                            f"{r['bars'][0]['seconds'] / max(b['seconds'], 1e-9):.2f}", r["cores"], r.get("when", "")])
    path = WORK_ROOT / "speedlab_results.zip"
    with zipfile.ZipFile(path, "w") as z:
        for p in files + [RESULTS_DIR / "results.csv"]:
            z.write(p, p.name)
    return str(path)


def version(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return "not installed"


def system_info():
    try:
        rustc = subprocess.run(["rustc", "--version"], capture_output=True, text=True).stdout.strip() or "not installed"
    except Exception:
        rustc = "not installed"
    try:
        import psutil
        ram = f"{psutil.virtual_memory().total / 2**30:.0f} GB total · {psutil.virtual_memory().available / 2**30:.0f} GB free"
    except Exception:
        ram = "?"
    rows = [("CPU cores", os.cpu_count()), ("RAM", ram), ("Python", sys.version.split()[0]), ("Rust", rustc),
            ("PyO3", PYO3_VERSION)]
    rows += [(p, version(p)) for p in ("polars", "pandas", "tokenizers", "tiktoken", "orjson", "safetensors", "torch",
                                        "uv", "numpy", "gradio")]
    return "| | |\n|---|---|\n" + "\n".join(f"| **{k}** | {v} |" for k, v in rows)

# ---------------------------------------------------------------------------
# 4) Interface
# ---------------------------------------------------------------------------
try:
    gr.close_all()
except Exception:
    pass

CSS = """
.gradio-container { max-width: 1200px !important; margin: auto !important; }
.brand-header { text-align:center; background: linear-gradient(135deg,#0f0c29 0%,#302b63 55%,#24243e 100%); padding:28px; border-radius:15px; margin-bottom:12px; box-shadow:0 10px 25px rgba(0,0,0,0.4); }
.brand-title { color:#fff; font-size:2em; font-weight:700; margin:0 0 6px 0; }
.brand-subtitle { color:#c0c0ff; font-size:1em; margin:0; font-weight:400; }
.info-chips { display:flex; justify-content:center; gap:8px; flex-wrap:wrap; margin-bottom:14px; }
.chip { background:rgba(124,92,255,0.12); color:#7c5cff; border:1px solid rgba(124,92,255,0.35); padding:5px 12px; border-radius:999px; font-size:13px; font-weight:600; }
button.primary { background: linear-gradient(135deg,#7c5cff 0%,#5b3fd9 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.rust-btn { background: linear-gradient(135deg,#F74C00 0%,#b33600 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.stop-btn { background: linear-gradient(135deg,#ef4444 0%,#b91c1c 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.clear-btn { background: linear-gradient(135deg,#6b7280 0%,#374151 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.status-box textarea { font-family: ui-monospace, monospace !important; font-size: 13px !important; }
.footer { text-align:center; padding:18px; margin-top:24px; border-top:1px solid rgba(128,128,128,0.25); color:#888; font-size:13px; }
.footer a { color:#7c5cff; }
.promo-links { display:flex; justify-content:center; gap:10px; flex-wrap:wrap; margin-top:16px; }
.promo-links a { text-decoration:none !important; color:#fff !important; font-weight:700; font-size:14px; padding:8px 16px; border-radius:999px; box-shadow:0 4px 14px rgba(0,0,0,0.3); transition:transform .15s; }
.promo-links a:hover { transform:translateY(-2px); }
.promo-yt { background:#FF0000; } .promo-gh { background:#24292f; border:1px solid #f5c518; } .promo-ig { background:linear-gradient(45deg,#f09433,#dc2743,#bc1888); } .promo-tt { background:#000; border:1px solid #25F4EE; }
.promo-card { display:flex; align-items:center; justify-content:space-between; gap:16px; flex-wrap:wrap; margin-top:18px; padding:18px 22px; border-radius:15px; background:linear-gradient(135deg,rgba(247,76,0,0.14),rgba(124,92,255,0.14)); border:1px solid rgba(124,92,255,0.35); }
.promo-card .promo-text { font-size:15px; line-height:1.5; }
.promo-card .promo-text b { font-size:17px; }
"""

HEADER = """
<div class='brand-header'>
  <div class='brand-title'>🦀 Rust vs 🐍 Python · Speed Lab</div>
  <div class='brand-subtitle'>Real AI tools, raced live: tokenizers • DataFrames • JSON • model files • pip vs uv • your own Rust code</div>
  <div class='promo-links'>
    <a class='promo-yt' href='https://www.youtube.com/@thebuildai?sub_confirmation=1' target='_blank'>▶ Subscribe on YouTube</a>
    <a class='promo-gh' href='https://github.com/cafermutluozkan/free-ai-notebooks' target='_blank'>⭐ Star the free notebooks on GitHub</a>
    <a class='promo-ig' href='https://www.instagram.com/thebuildai/' target='_blank'>📸 Instagram</a>
    <a class='promo-tt' href='https://www.tiktok.com/@the.build.ai' target='_blank'>🎵 TikTok</a>
  </div>
</div>
"""
PROMO = """
<div class='promo-card'>
  <div class='promo-text'><b>🎉 Enjoying the Speed Lab?</b><br>
  It's free, like all my notebooks. <b>Subscribe</b> for a new free AI tool every week, and <b>⭐ star the GitHub repo</b>
  so more people can find it. Post your numbers in the YouTube comments!</div>
  <div class='promo-links' style='margin-top:0'>
    <a class='promo-yt' href='https://www.youtube.com/@thebuildai?sub_confirmation=1' target='_blank'>▶ Subscribe</a>
    <a class='promo-gh' href='https://github.com/cafermutluozkan/free-ai-notebooks' target='_blank'>⭐ Star on GitHub</a>
  </div>
</div>
"""
chips = [f"🖥️ {os.cpu_count()} CPU cores", "🚫 No GPU needed", f"🐍 Python {sys.version.split()[0]}",
         f"🦀 Rust compiler: {'ready' if rust_toolchain_ok() else 'off'}", "✅ Every result is checked",
         "📌 Pinned versions"]
CHIPS = "<div class='info-chips'>" + "".join(f"<span class='chip'>{c}</span>" for c in chips) + "</div>"
FOOTER = ("<div class='footer'>Built by <a href='https://www.thebuildai.tech/' target='_blank'><b>TheBuildAI</b></a> · "
          "<a href='https://www.youtube.com/@thebuildai?sub_confirmation=1' target='_blank'>YouTube</a> · "
          "<a href='https://github.com/cafermutluozkan/free-ai-notebooks' target='_blank'>⭐ GitHub</a> · "
          "<a href='https://www.instagram.com/thebuildai/' target='_blank'>Instagram</a> · "
          "<a href='https://www.tiktok.com/@the.build.ai' target='_blank'>TikTok</a> · "
          "Numbers depend on the machine: run it yourself and compare.</div>")
first_test = list(TESTS)[0]

with gr.Blocks(title="Rust vs Python Speed Lab — TheBuildAI") as demo:
    gr.HTML(HEADER)
    gr.HTML(CHIPS)
    with gr.Tabs():
        # ---- Arena ----
        with gr.Tab("🏁 Speed Arena"):
            with gr.Row():
                with gr.Column(scale=2):
                    a_test = gr.Radio(list(TESTS), value=first_test, label="Pick a race")
                    a_info = gr.Markdown(TESTS[first_test][1])
                    a_size = gr.Radio(SIZES, value=DEFAULT_SIZE, label="Size",
                                      info="Quick = seconds · Standard = the video numbers · Heavy = bigger data")
                    with gr.Row():
                        a_go = gr.Button("▶ Run race", variant="primary", size="lg")
                        a_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
                        a_clear = gr.Button("🗑️ Clear", size="lg", elem_classes="clear-btn")
                    a_status = gr.Textbox(label="Status", lines=2, interactive=False, elem_classes="status-box")
                with gr.Column(scale=3):
                    a_chart = gr.Image(label="Result", type="filepath", height=440)
            a_result = gr.Markdown()
            with gr.Accordion("📜 Live log", open=False):
                a_log = gr.Textbox(lines=12, show_label=False, interactive=False, autoscroll=True,
                                   elem_classes="status-box")

        # ---- Write Rust ----
        with gr.Tab("🦀 Write Rust"):
            gr.Markdown("This is **real Rust**. [PyO3](https://pyo3.rs) turns it into a Python module, and Python calls "
                        "it like any other function. Edit it (make it faster, or break it on purpose to see how the "
                        "compiler explains the error), then press **Compile & Race**. The first compile already "
                        "happened in Step 2, so this takes a few seconds.")
            with gr.Row():
                r_race = gr.Radio(list(RACES), value=list(RACES)[0], label="Race")
                r_size = gr.Radio(SIZES, value=DEFAULT_SIZE, label="Size")
            with gr.Row():
                r_py = gr.Code(PY_CODE["rust_primes"], language="python", label="🐍 Python (the same algorithm)",
                               interactive=False, lines=24)
                r_code = gr.Code(RUST_SOURCE, language="cpp", label="🦀 Rust · src/lib.rs (you can edit it)",
                                 interactive=True, lines=24)
            with gr.Row():
                r_go = gr.Button("🔨 Compile & Race", size="lg", elem_classes="rust-btn")
                r_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
                r_reset = gr.Button("↺ Reset Rust code", size="lg", elem_classes="clear-btn")
            r_status = gr.Textbox(label="Status", lines=1, interactive=False, elem_classes="status-box")
            with gr.Row():
                r_chart = gr.Image(label="Result", type="filepath", height=440)
                r_result = gr.Markdown()
            with gr.Accordion("🦀 Compiler output (cargo build --release)", open=False):
                r_log = gr.Textbox(lines=14, show_label=False, interactive=False, autoscroll=True,
                                   elem_classes="status-box")

        # ---- Scoreboard ----
        with gr.Tab("🏆 Scoreboard"):
            gr.Markdown("Runs **every race one after another** and builds one summary chart. Each chart is saved to "
                        f"`{RESULTS_DIR}` as a 16:9 PNG (ready for a video or a post).")
            with gr.Row():
                s_size = gr.Radio(SIZES, value=DEFAULT_SIZE, label="Size")
                s_go = gr.Button("▶ Run all tests", variant="primary", size="lg")
                s_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
            s_status = gr.Textbox(label="Status", lines=1, interactive=False, elem_classes="status-box")
            s_chart = gr.Image(label="Scoreboard", type="filepath", height=520)
            s_table = gr.Markdown(scoreboard_table())
            s_gallery = gr.Gallery(label="All charts", columns=3, height=420, object_fit="contain")
            with gr.Row():
                s_zip = gr.Button("📦 Download all results (ZIP: charts + JSON + CSV)", size="sm")
                s_file = gr.File(label="Results ZIP", height=80)

        # ---- System ----
        with gr.Tab("💻 System"):
            y_info = gr.Markdown(system_info())
            y_refresh = gr.Button("🔄 Refresh", size="sm")

    gr.HTML(PROMO)
    gr.HTML(FOOTER)

    # only one race at a time (two at once would slow each other down and ruin the numbers)
    ONE_AT_A_TIME = dict(concurrency_id="bench", concurrency_limit=1)
    a_test.change(lambda t: TESTS[t][1], a_test, a_info)
    a_event = a_go.click(arena, [a_test, a_size], [a_chart, a_result, a_status, a_log], **ONE_AT_A_TIME)
    a_stop.click(stop, outputs=a_status, cancels=[a_event], concurrency_limit=None)
    a_clear.click(lambda: (None, "", "", ""), outputs=[a_chart, a_result, a_status, a_log])

    r_race.change(lambda r: PY_CODE[RACES[r]], r_race, r_py)
    r_event = r_go.click(compile_and_race, [r_code, r_race, r_size], [r_chart, r_result, r_status, r_log],
                         **ONE_AT_A_TIME)
    r_stop.click(stop, outputs=r_status, cancels=[r_event], concurrency_limit=None)
    r_reset.click(lambda: RUST_SOURCE, outputs=r_code)

    s_event = s_go.click(run_all, s_size, [s_chart, s_table, s_gallery, s_status], **ONE_AT_A_TIME)
    s_stop.click(stop, outputs=s_status, cancels=[s_event], concurrency_limit=None)
    s_zip.click(zip_results, outputs=s_file)
    y_refresh.click(system_info, outputs=y_info)
    demo.load(lambda: (draw_scoreboard(), gallery()), outputs=[s_chart, s_gallery])

print("Launching Gradio interface... (the public link appears in ~10-20 s)")
demo.queue()
demo.launch(share=SHARE, inline=False, debug=True, css=CSS, theme=gr.themes.Soft(primary_hue="violet"),
            allowed_paths=[str(RESULTS_DIR), str(WORK_ROOT)])

---
## Step 5: Diagnostics

If something goes wrong, stop Step 4 with ■ and run this. It shows the Rust compiler, the last compiler output and the saved results.

In [ ]:
# Step 5 - If something goes wrong: diagnostics (stop Step 4 with ■ first)
import json
import subprocess

for cmd in (["rustc", "--version"], ["cargo", "--version"]):
    try:
        print(subprocess.run(cmd, capture_output=True, text=True).stdout.strip())
    except FileNotFoundError:
        print(cmd[0], "is not installed (INSTALL_RUST = True in Step 1, then run Step 2)")

print("\nCompiled Rust module:", [p.name for p in RUSTMOD_DIR.glob("speedrust.*")] or "none yet")
if (RUST_DIR / "Cargo.toml").exists():
    print("\n--- cargo build --release (last 30 lines) ---")
    r = subprocess.run(["cargo", "build", "--release"], cwd=RUST_DIR, capture_output=True, text=True,
                       env=dict(os.environ, PYO3_PYTHON=sys.executable, PYO3_BUILD_EXTENSION_MODULE="1"))
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))

results = RESULTS_DIR / "results.json"
if results.exists():
    print("\n--- saved results ---")
    for r in json.loads(results.read_text()):
        print(f"{r['test']:<12} {r['size']:<9} {r['headline']}  {r['check'][:1]}")

---
## Credits & Licenses

This notebook only **downloads** the tools and data below from their official sources and runs them inside your own session. It does not host or redistribute them.

| Component | What it is used for | License |
|---|---|---|
| **Rust** + **Cargo** | compiles your Rust code | [MIT / Apache-2.0](https://www.rust-lang.org/policies/licenses) |
| **PyO3** | lets Python call Rust | [MIT / Apache-2.0](https://github.com/PyO3/pyo3) |
| **Hugging Face tokenizers** | Rust tokenizer | [Apache-2.0](https://github.com/huggingface/tokenizers/blob/main/LICENSE) |
| **tiktoken** by OpenAI | Rust tokenizer | [MIT](https://github.com/openai/tiktoken/blob/main/LICENSE) |
| **GPT-2 tokenizer files** ([openai-community/gpt2](https://huggingface.co/openai-community/gpt2)) | vocabulary + merges; the pure-Python BPE follows OpenAI's original GPT-2 encoder | [MIT](https://github.com/openai/gpt-2/blob/master/LICENSE) |
| **WikiText-2** ([Salesforce/wikitext](https://huggingface.co/datasets/Salesforce/wikitext)) | Wikipedia text to tokenize | [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/) |
| **polars** / **pandas** | DataFrames | [MIT](https://github.com/pola-rs/polars/blob/main/LICENSE) / [BSD-3-Clause](https://github.com/pandas-dev/pandas/blob/main/LICENSE) |
| **orjson** | Rust JSON | [Apache-2.0 / MIT](https://github.com/ijl/orjson) |
| **safetensors** / **PyTorch** | model files | [Apache-2.0](https://github.com/huggingface/safetensors/blob/main/LICENSE) / [BSD-3-Clause](https://github.com/pytorch/pytorch/blob/main/LICENSE) |
| **uv** by Astral / **pip** | installing packages | [Apache-2.0 / MIT](https://github.com/astral-sh/uv) / [MIT](https://github.com/pypa/pip/blob/main/LICENSE.txt) |
| **NumPy** · **Matplotlib** · **Gradio** | reference, charts, web page | [BSD-3-Clause](https://numpy.org/doc/stable/license.html) · [Matplotlib license](https://matplotlib.org/stable/project/license.html) · [Apache-2.0](https://www.apache.org/licenses/LICENSE-2.0) |

**About fairness:** the timings come from your own session and change from machine to machine. Every race checks that both sides produce the same result, data creation is never timed, and the notes under each chart say where the comparison has limits (for example, pandas already runs C code, and safetensors wins by reading lazily).

---

<div align='center'>

### 🎉 Enjoyed this notebook?

If this was helpful, please **upvote**, subscribe for more free AI tools, and ⭐ **star [the free notebooks repo](https://github.com/cafermutluozkan/free-ai-notebooks)** on GitHub!

  <a href='https://www.youtube.com/@thebuildai?sub_confirmation=1'>
    <img src='https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white' />
  </a>
  <a href='https://www.instagram.com/thebuildai/'>
    <img src='https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white' />
  </a>
  <a href='https://www.tiktok.com/@the.build.ai'>
    <img src='https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white' />
  </a>
  <a href='https://github.com/cafermutluozkan'>
    <img src='https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white' />
  </a>

  <p style='color: #888; margin-top: 15px;'>🦀 Rust vs Python Speed Lab - Built by <strong><a href='https://www.thebuildai.tech/'>TheBuildAI</a></strong> 🌍</p>

</div>